# Del pozo a realizaciones condicionadas
Implementación pedagógica de SGS en un corte 2D pequeño. En un proyecto 3D real deben revisarse soporte, anisotropía, vecindario, tendencia y transformación normal-score.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import cdist
rng = np.random.default_rng(20261027)
nx, ny = 24, 16
xx, yy = np.meshgrid(np.arange(nx), np.arange(ny))
grid = np.c_[xx.ravel(), yy.ravel()]
# Residuos gaussianizados sintéticos observados en 18 celdas.
obs_idx = rng.choice(len(grid), 18, replace=False)
obs_xy = grid[obs_idx]
def cov(a,b, alcance=6.): return np.exp(-cdist(a,b)/alcance)
obs_y = rng.multivariate_normal(np.zeros(len(obs_idx)), cov(obs_xy,obs_xy)+1e-8*np.eye(len(obs_idx)))

In [ ]:
def sgs(grid, obs_xy, obs_y, semilla, vecinos=12):
    rg = np.random.default_rng(semilla)
    valores = np.full(len(grid), np.nan)
    conocidos_xy = obs_xy.astype(float).copy()
    conocidos_y = obs_y.astype(float).copy()
    # Si una observación cae en la grilla, se honra exactamente.
    mapa = {tuple(p):v for p,v in zip(obs_xy,obs_y)}
    pendientes = [i for i,p in enumerate(grid) if tuple(p) not in mapa]
    for i,p in enumerate(grid):
        if tuple(p) in mapa: valores[i] = mapa[tuple(p)]
    rg.shuffle(pendientes)
    for i in pendientes:
        p = grid[i:i+1]
        dist = cdist(p, conocidos_xy).ravel()
        use = np.argsort(dist)[:vecinos]
        xy, y = conocidos_xy[use], conocidos_y[use]
        K = cov(xy,xy) + 1e-7*np.eye(len(use))
        k = cov(xy,p).ravel()
        w = np.linalg.solve(K,k)
        media = w@y
        var = max(1.-k@w, 1e-8)
        valores[i] = rg.normal(media, np.sqrt(var))
        conocidos_xy = np.vstack([conocidos_xy,p])
        conocidos_y = np.r_[conocidos_y,valores[i]]
    return valores.reshape(ny,nx)
real = np.array([sgs(grid,obs_xy,obs_y,1000+r) for r in range(24)])
np.max([abs(real[r, int(y), int(x)]-v) for r in range(len(real)) for (x,y),v in zip(obs_xy,obs_y)])

In [ ]:
fig, ax = plt.subplots(2,3,figsize=(11,6),constrained_layout=True)
for j in range(4):
    im=ax.flat[j].imshow(real[j],origin='lower',cmap='RdBu_r',vmin=-2.5,vmax=2.5)
    ax.flat[j].scatter(obs_xy[:,0],obs_xy[:,1],c=obs_y,cmap='RdBu_r',vmin=-2.5,vmax=2.5,edgecolor='k')
    ax.flat[j].set_title(f'Realización {j+1}')
ax.flat[4].imshow(real.mean(0),origin='lower',cmap='RdBu_r',vmin=-1.5,vmax=1.5); ax.flat[4].set_title('Media tipo E')
ax.flat[5].imshow((real>0.5).mean(0),origin='lower',cmap='Blues',vmin=0,vmax=1); ax.flat[5].set_title('P(residuo > 0.5)')
for a in ax.flat: a.set(xticks=[],yticks=[])
plt.show()

## Lectura crítica
El código usa media global cero, covarianza exponencial isotrópica y observaciones ya gaussianizadas. No sustituye el análisis de tendencia, el ajuste de variogramas direccionales, la transformación inversa ni la validación de un caso real.